# Notebook 2 — PredCount on cross-hyperedge triangles

Notebook 1 settled the negative half: on 85 real measurements $\alpha_H/\kappa$ never left
$[0.60, 1.00]$, so predictions cannot buy an exponent on any real domain tested. This notebook
measures what they *do* buy, on a primitive that is new to the streaming-with-predictions
literature: **cross-hyperedge triangles** — triples $\{u,v,w\}$ pairwise connected in the projection
of a hypergraph but never co-occurring in a single hyperedge, i.e. genuine higher-order triadic
closure.

**Separating the two mechanisms.** The gain over the Fichtenberger–Peng sampler has two sources and
they must be reported apart, because only the second is about prediction:

| quantity | what it isolates |
|---|---|
| `loc_x` | replacing the global $\sqrt{2m}$ closing rule by the local normalizer $D_x$ — **localization**, with $\hat w \equiv 0$ |
| `perfect_over_loc` | what a perfect predictor adds *on top of* localization |
| `permuted_over_loc` | the same weight multiset shuffled across edges — the distributional null |
| `mindeg_over_loc` | the cheap Tonic-style min-endpoint-degree predictor |

Reporting `perfect_x` alone would conflate the two and would overlap with the localization paper
already under review. Every table below reports all four.

All numbers are exact success probabilities summed over the sampling distribution — no Monte Carlo.

## 0. Library

In [ ]:
"""Core measurement library: kappa, alpha_H, copy-bearing subgraphs, diagnostics."""
import gzip
import io
import os
import urllib.request
from collections import defaultdict

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_flow

# ----------------------------------------------------------------- graph basics


def relabel(edges):
    """Map arbitrary hashable node labels to consecutive integers."""
    ids, out = {}, []
    for u, v in edges:
        for x in (u, v):
            if x not in ids:
                ids[x] = len(ids)
        out.append((ids[u], ids[v]))
    return out, ids


def canon(edges):
    """De-duplicate, drop self-loops, relabel to ints, return sorted (u,v), u<v."""
    edges, _ = relabel(edges)
    s = set()
    for u, v in edges:
        if u == v:
            continue
        s.add((u, v) if u < v else (v, u))
    return sorted(s)


def build_adj(edges):
    adj = defaultdict(set)
    for u, v in edges:
        if u == v:
            continue
        adj[u].add(v)
        adj[v].add(u)
    return adj


def degeneracy(adj):
    """Exact degeneracy (k-core number) by peeling.  O(n + m)."""
    if not adj:
        return 0, {}
    deg = {v: len(adj[v]) for v in adj}
    maxdeg = max(deg.values())
    buckets = [set() for _ in range(maxdeg + 1)]
    for v, d in deg.items():
        buckets[d].add(v)
    core, k, removed = {}, 0, set()
    for _ in range(len(deg)):
        i = 0
        while i <= maxdeg and not buckets[i]:
            i += 1
        if i > maxdeg:
            break
        v = buckets[i].pop()
        k = max(k, i)
        core[v] = k
        removed.add(v)
        for w in adj[v]:
            if w in removed:
                continue
            d = deg[w]
            buckets[d].discard(w)
            deg[w] = d - 1
            buckets[d - 1].add(w)
    return k, core


# -------------------------------------------------- oracle-width (pseudoarboricity)


def _orientation_feasible(edges, nodes, k):
    """Does an orientation with max out-degree <= k exist?  Max-flow feasibility test.

    s -> edge (cap 1); edge -> each endpoint (cap 1); vertex -> t (cap k).
    Feasible iff maxflow == |E|  (Hakimi / Frank-Gyarfas).
    """
    m, n = len(edges), len(nodes)
    if m == 0:
        return True
    idx = {v: i for i, v in enumerate(nodes)}
    S, E0, V0 = 0, 1, 1 + m
    T = 1 + m + n
    rows, cols, data = [], [], []
    for i, (u, v) in enumerate(edges):
        rows.append(S); cols.append(E0 + i); data.append(1)
        rows.append(E0 + i); cols.append(V0 + idx[u]); data.append(1)
        rows.append(E0 + i); cols.append(V0 + idx[v]); data.append(1)
    for i in range(n):
        rows.append(V0 + i); cols.append(T); data.append(int(k))
    g = csr_matrix((np.array(data, dtype=np.int32),
                    (np.array(rows), np.array(cols))), shape=(T + 1, T + 1))
    return int(maximum_flow(g, S, T).flow_value) == m


def pseudoarboricity(edges):
    """Exact min-max-out-degree orientation number of an edge set."""
    edges = canon(edges)
    if not edges:
        return 0
    nodes = sorted({x for e in edges for x in e})
    hi = max(degeneracy(build_adj(edges))[0], 1)
    lo = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if _orientation_feasible(edges, nodes, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


# --------------------------------------------------------------- copy-bearing sets


def copy_bearing_K3(adj):
    """Edges on >= 1 triangle, plus #K3."""
    out, tri = [], 0
    for u in adj:
        for v in adj[u]:
            if u >= v:
                continue
            c = len(adj[u] & adj[v])
            if c:
                out.append((u, v))
                tri += c
    return out, tri // 3


def codegree_map(adj, deg_cap=None):
    """codeg[(a,b)] = |N(a) & N(b)| for pairs with a common neighbour."""
    codeg = defaultdict(int)
    for w in adj:
        nb = sorted(adj[w])
        if deg_cap is not None and len(nb) > deg_cap:
            continue
        for i in range(len(nb)):
            a = nb[i]
            for j in range(i + 1, len(nb)):
                codeg[(a, nb[j])] += 1
    return codeg


def copy_bearing_C4(adj, deg_cap=None):
    """Edges on >= 1 four-cycle (exact), plus #C4.

    Edge (u,v) lies on a C4 iff some u' in N(v)\\{u} has |N(u) & N(u')| >= 2.
    """
    codeg = codegree_map(adj, deg_cap)
    heavy = {p for p, c in codeg.items() if c >= 2}
    n_c4 = sum(c * (c - 1) // 2 for c in codeg.values()) // 2
    out = []
    for u in adj:
        for v in adj[u]:
            if u >= v:
                continue
            for up in adj[v]:
                if up == u:
                    continue
                p = (u, up) if u < up else (up, u)
                if p in heavy:
                    out.append((u, v))
                    break
    return out, n_c4


# ------------------------------------------------------------------- diagnostic


def diagnostic(edges, pattern="K3", label="", deg_cap=None, exact_alpha=True):
    """Return the full row: n, m, kappa, kappa_copy, alpha_H, ratios, copy density."""
    edges = canon(edges)
    adj = build_adj(edges)
    kappa, _ = degeneracy(adj)
    if pattern == "K3":
        cb, n_copies = copy_bearing_K3(adj)
    elif pattern == "C4":
        cb, n_copies = copy_bearing_C4(adj, deg_cap)
    else:
        raise ValueError(pattern)
    adj_cb = build_adj(cb)
    kappa_copy, _ = degeneracy(adj_cb)
    if exact_alpha:
        alpha = pseudoarboricity(cb)
    else:  # cheap two-sided bracket, no max-flow
        alpha = None
    row = dict(
        label=label, pattern=pattern, n=len(adj), m=len(edges),
        kappa=kappa, kappa_copy=kappa_copy, alpha=alpha,
        m_copy=len(cb), n_copies=n_copies,
        copy_density=(n_copies / len(edges)) if edges else 0.0,
        alpha_over_kappa=(alpha / kappa) if (alpha is not None and kappa) else None,
        kappa_copy_over_kappa=(kappa_copy / kappa) if kappa else None,
        # structural bracket:  ceil(kappa_copy/2) <= alpha_H <= kappa_copy <= kappa
        alpha_lb=int(np.ceil(kappa_copy / 2)),
        alpha_ub=kappa_copy,
    )
    return row


# ------------------------------------------------------------------ constructions


def affine_plane_incidence(q):
    """Incidence graph of AG(2,q), q prime.  C4-free, degeneracy Theta(q)=Theta(sqrt(m))."""
    E = []
    for a in range(q):
        for b in range(q):
            lid = ("L", a, b)
            for x in range(q):
                E.append((("P", x, (a * x + b) % q), lid))
    for c in range(q):
        lid = ("V", c)
        for y in range(q):
            E.append((("P", c, y), lid))
    return E


def steiner_plus_butterflies(q, n_gadgets, r=3):
    """AG(2,q) copy-free dense core  +  n_gadgets shallow butterfly gadgets.

    Natural analogue of the friendship+K_{d,d} instance of Theorem 11, with the
    copy-free dense region replaced by a genuine partial linear space.
    """
    E = affine_plane_incidence(q)
    for g in range(n_gadgets):
        a, b = ("G", g, 0), ("G", g, 1)
        for t in range(2):
            h = ("H", g, t)
            E.append((a, h))
            E.append((b, h))
            for j in range(r - 2):
                E.append((("F", g, t, j), h))
    return E


def friendship_plus_decoy(s):
    """The instance of Theorem 11: F_s  U  K_{d,d}, d = ceil(sqrt(s))."""
    d = int(np.ceil(np.sqrt(s)))
    E = []
    for i in range(s):
        E += [("h", ("a", i)), ("h", ("b", i)), (("a", i), ("b", i))]
    for i in range(d):
        for j in range(d):
            E.append((("L", i), ("R", j)))
    return E


def random_uniform_hypergraph(n, m_h, r, seed=0):
    rng = np.random.default_rng(seed)
    return [tuple(rng.choice(n, size=r, replace=False)) for _ in range(m_h)]


def incidence_edges(hyperedges):
    """Bipartite incidence graph of a hyperedge list."""
    E = []
    for i, h in enumerate(hyperedges):
        for v in set(h):
            E.append((("v", v), ("e", i)))
    return E


# ----------------------------------------------------------------------- loaders


def fetch(url, dest):
    if os.path.exists(dest):
        return dest
    urllib.request.urlretrieve(url, dest)
    return dest


def load_temporal(path):
    """SNAP temporal edge list: 'u v t' per line.  Returns sorted (t,u,v) array."""
    op = gzip.open if path.endswith(".gz") else open
    rows = []
    with op(path, "rt") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            p = line.split()
            if len(p) < 3:
                continue
            rows.append((int(p[2]), int(p[0]), int(p[1])))
    rows.sort()
    return rows


def load_hyperedges_simplices(nverts_path, simplices_path):
    """Benson ARB format: one size per line + a flat vertex stream."""
    op1 = gzip.open if nverts_path.endswith(".gz") else open
    op2 = gzip.open if simplices_path.endswith(".gz") else open
    sizes = [int(x) for x in op1(nverts_path, "rt").read().split()]
    flat = [int(x) for x in op2(simplices_path, "rt").read().split()]
    out, i = [], 0
    for s in sizes:
        out.append(tuple(flat[i:i + s]))
        i += s
    return out


def load_hyperedges_lines(path):
    """One hyperedge per line, whitespace- or comma-separated vertex ids."""
    op = gzip.open if path.endswith(".gz") else open
    out = []
    with op(path, "rt") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            out.append(tuple(line.replace(",", " ").split()))
    return out


# ------------------------------------- higher-order pattern: cross-hyperedge triangles

def projection(hyperedges, max_size=25):
    """Projected graph of a hypergraph + the membership map v -> set of hyperedge ids."""
    memb = defaultdict(set)
    E = set()
    for i, h in enumerate(hyperedges):
        s = sorted(set(h))
        if len(s) < 2 or len(s) > max_size:
            continue
        for v in s:
            memb[v].add(i)
        for a in range(len(s)):
            for b in range(a + 1, len(s)):
                E.add((s[a], s[b]))
    return sorted(E), memb


def copy_bearing_cross_K3(adj, memb):
    """Edges on >= 1 triangle NOT contained in a single hyperedge, plus the count.

    A triangle {u,v,w} is *cross* iff memb[u] & memb[v] & memb[w] is empty: the three
    vertices never co-occur in one hyperedge, so the triangle is genuine higher-order
    triadic closure rather than the clique a single hyperedge drops on the projection.
    """
    out, n_cross = [], 0
    for u in adj:
        for v in adj[u]:
            if u >= v:
                continue
            common = adj[u] & adj[v]
            if not common:
                continue
            muv = memb[u] & memb[v]
            hit = False
            for w in common:
                if not (muv & memb[w]):
                    n_cross += 1
                    hit = True
            if hit:
                out.append((u, v))
    return out, n_cross // 3


def diagnostic_hypergraph(hyperedges, label='', max_size=25, exact_alpha=True):
    """Diagnostic for cross-hyperedge triangles on the projected graph."""
    E, memb = projection(hyperedges, max_size)
    adj = build_adj(E)
    kappa, _ = degeneracy(adj)
    cb, n_cross = copy_bearing_cross_K3(adj, memb)
    kappa_copy, _ = degeneracy(build_adj(cb))
    alpha = pseudoarboricity(cb) if exact_alpha else None
    return dict(
        label=label, pattern='cross-K3', n=len(adj), m=len(E),
        kappa=kappa, kappa_copy=kappa_copy, alpha=alpha,
        m_copy=len(cb), n_copies=n_cross,
        copy_density=n_cross / len(E) if E else 0.0,
        alpha_over_kappa=(alpha / kappa) if (alpha is not None and kappa) else None,
        kappa_copy_over_kappa=(kappa_copy / kappa) if kappa else None,
        alpha_lb=int(np.ceil(kappa_copy / 2)), alpha_ub=kappa_copy,
        max_hyperedge=max((len(set(h)) for h in hyperedges), default=0),
    )


def cliques_plus_cross(n_cliques, clique_size, n_gadgets, seed=0):
    """Synthetic positive control: big copy-free cliques + shallow cross-triangle gadgets."""
    rng = np.random.default_rng(seed)
    H, nxt = [], 0
    cliques = []
    for _ in range(n_cliques):
        c = list(range(nxt, nxt + clique_size)); nxt += clique_size
        H.append(tuple(c)); cliques.append(c)
    for g in range(n_gadgets):
        c = cliques[rng.integers(len(cliques))]
        u, v = rng.choice(c, size=2, replace=False)
        w = nxt; nxt += 1
        H.append((int(u), w))
        H.append((int(v), w))
    return H


"""PredCount for cross-hyperedge triangles: exact success probabilities and speedups."""
import numpy as np
from collections import defaultdict


class CrossProjection:
    """Projected graph of a hypergraph with exact per-edge cross-triangle counts."""

    def __init__(self, hyperedges, max_size=25):
        hs = [tuple(sorted(set(h))) for h in hyperedges]
        hs = [h for h in hs if 2 <= len(h) <= max_size]
        hs = list(dict.fromkeys(hs))
        verts = sorted({v for h in hs for v in h})
        self.idx = {v: i for i, v in enumerate(verts)}
        self.n = len(verts)
        self.H = [sum(1 << self.idx[v] for v in h) for h in hs]
        self.memb = [set() for _ in range(self.n)]
        adj = [0] * self.n
        for hid, h in enumerate(hs):
            ids = [self.idx[v] for v in h]
            for a in ids:
                self.memb[a].add(hid)
            bits = self.H[hid]
            for a in ids:
                adj[a] |= bits & ~(1 << a)
        self.adj = adj
        self.edges = []
        for u in range(self.n):
            b = adj[u] >> (u + 1)
            v = u + 1
            while b:
                if b & 1:
                    self.edges.append((u, v))
                b >>= 1
                v += 1
        self.m = len(self.edges)
        self.deg = [bin(a).count('1') for a in adj]
        # (deg, id) ordering used by the canonical discovery path
        order = sorted(range(self.n), key=lambda v: (self.deg[v], v))
        self.rank = [0] * self.n
        for r, v in enumerate(order):
            self.rank[v] = r
        self._cross_counts()

    def _covered(self, u, v):
        cov = 0
        for h in self.memb[u] & self.memb[v]:
            cov |= self.H[h]
        return cov

    def _cross_counts(self):
        """t_cross(e) for every edge, via  |N(u)&N(v)|  minus the hyperedge-covered part."""
        self.t = {}
        self.cross_partners = {}
        total = 0
        for (u, v) in self.edges:
            common = self.adj[u] & self.adj[v]
            cov = self._covered(u, v)
            cross = common & ~cov
            c = bin(cross).count('1')
            self.t[(u, v)] = c
            self.cross_partners[(u, v)] = cross
            total += c
        self.n_cross = total // 3

    # ---------------------------------------------------------------- predictors
    def weights_perfect(self):
        return dict(self.t)

    def weights_permuted(self, seed=0):
        rng = np.random.default_rng(seed)
        vals = list(self.t.values())
        rng.shuffle(vals)
        return dict(zip(self.t.keys(), vals))

    def weights_mindeg(self):
        return {e: min(self.deg[e[0]], self.deg[e[1]]) for e in self.t}

    def weights_uniform(self):
        return {e: 0 for e in self.t}

    # ------------------------------------------------------------ success probs
    def success_prob(self, w, first_edge_only=False):
        """Exact sum over cross triangles of the weighted per-copy path probability."""
        W = sum(w[e] + 1 for e in self.t)
        D = [0.0] * self.n
        for (u, v) in self.edges:
            D[u] += w[(u, v)] + 1
            D[v] += w[(u, v)] + 1
        p = 0.0
        for (a, b) in self.edges:
            cross = self.cross_partners[(a, b)]
            if not cross:
                continue
            ra, rb = self.rank[a], self.rank[b]
            hi = max(ra, rb)
            x, y = (a, b) if ra < rb else (b, a)
            z = 0
            c = cross
            while c:
                if c & 1:
                    if self.rank[z] > hi:            # (a,b) is the canonical base edge
                        exy = (min(x, y), max(x, y))
                        exz = (min(x, z), max(x, z))
                        if first_edge_only:
                            p += (w[exy] + 1) / W * (1.0 / self.deg[x])
                        else:
                            p += (w[exy] + 1) / W * (w[exz] + 1) / D[x]
                c >>= 1
                z += 1
        return p

    def baseline_prob(self):
        """Prediction-free sampler: every copy found with probability (2m)^{-3/2}."""
        return self.n_cross / (2 * self.m) ** 1.5

    def speedup(self, w, **kw):
        return self.success_prob(w, **kw) / self.baseline_prob()


    def report(self, seed=0):
        """All four numbers: baseline, localization only, predictor, permutation control."""
        p_base = self.baseline_prob()
        p_loc = self.success_prob(self.weights_uniform())
        p_perf = self.success_prob(self.weights_perfect())
        p_perm = self.success_prob(self.weights_permuted(seed))
        p_mind = self.success_prob(self.weights_mindeg())
        p_first = self.success_prob(self.weights_perfect(), first_edge_only=True)
        s = lambda p: p / p_base
        sl = lambda p: p / p_loc
        gain = sl(p_perf) - 1.0
        struct = ((sl(p_perf) - sl(p_perm)) / gain) if gain > 1e-12 else float('nan')
        return dict(
            n=self.n, m=self.m, n_cross=self.n_cross,
            copy_density=self.n_cross / self.m,
            loc_x=s(p_loc),            # localization alone, vs the Fichtenberger-Peng sampler
            perfect_x=s(p_perf),       # localization + perfect predictor
            perfect_over_loc=sl(p_perf),   # predictor's own contribution
            permuted_over_loc=sl(p_perm),
            mindeg_over_loc=sl(p_mind),
            firstedge_x=s(p_first),
            structural_share=struct,
        )


## 1. Data

Same five hypergraphs as notebook 1. `MAX_SIMPLICES` caps each dataset: the cost is driven by the
number of cross triangles, and `congress-bills` has about 35M of them at full size. Raise the caps
if the runtime is comfortable — the measurement is exact either way, just on a sub-hypergraph.

In [ ]:
!pip -q install gdown

import gdown, tarfile, glob, os, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 220)

ARB_IDS = {
    'NDC-classes':            '1tpDiP1c73O18gCYEx4OI7kx8V_IdYxLt',
    'contact-primary-school': '1sBHSEIyvVKavAho524Ro4cKL66W6rn-t',
    'email-Enron':            '1tTVZkdpgRW47WWmsrdUCukHz0x2M6N77',
    'tags-math-sx':           '1eDevpF6EZs19rLouNpiKGLIlFOLUfKKG',
    'congress-bills':         '1gH1uJMZpn_SCJSRbORPH4JRQeevLwTyO',
    'DAWN':                   '1wGwoG7oBWnNN7J9TEpjqNpODbsYfMxp4',
}

def fetch_arb(name):
    tgz = f'{name}.tar.gz'
    try:
        if not os.path.exists(tgz):
            gdown.download(id=ARB_IDS[name], output=tgz, quiet=True)
        with tarfile.open(tgz) as t:
            t.extractall('.')
        nv = glob.glob(f'**/{name}-nverts.txt', recursive=True)
        sp = glob.glob(f'**/{name}-simplices.txt', recursive=True)
        return load_hyperedges_simplices(nv[0], sp[0])
    except Exception as e:
        print(f'  {name}: {type(e).__name__} - {e}')
        return None

# cap chosen so each dataset stays in the few-minutes range
MAX_SIMPLICES = {
    'contact-primary-school': 13000,
    'email-Enron':            2000,
    'NDC-classes':            1500,
    'DAWN':                   6000,
    'congress-bills':         3000,
}

HYPER = {}
for name in MAX_SIMPLICES:
    h = fetch_arb(name)
    if h:
        uniq = list(dict.fromkeys(tuple(sorted(set(x))) for x in h))
        uniq = [x for x in uniq if 2 <= len(x) <= 25][:MAX_SIMPLICES[name]]
        HYPER[name] = uniq
        print(f'{name}: {len(uniq)} unique simplices kept')

## 2. The measurement

`CrossProjection` builds the projected graph, computes the exact per-edge cross-triangle count
$t(e)$ with a bitset identity (common neighbours minus the hyperedge-covered part, no triangle
enumeration), and then sums the per-copy path probability over every cross triangle under each
weighting.

In [ ]:
rows = []
for name, h in HYPER.items():
    t0 = time.time()
    P = CrossProjection(h, max_size=25)
    r = P.report(seed=0)
    r['label'] = name
    r['secs'] = round(time.time() - t0, 1)
    rows.append(r)
    print(f"{name:24s} m={r['m']:>7d}  #cross={r['n_cross']:>9d}  "
          f"loc={r['loc_x']:.1f}x  pred={r['perfect_over_loc']:.2f}x  "
          f"perm={r['permuted_over_loc']:.2f}x  ({r['secs']}s)")

D = pd.DataFrame(rows)
D[['label','m','n_cross','copy_density','loc_x','perfect_x','perfect_over_loc',
   'permuted_over_loc','mindeg_over_loc','structural_share','firstedge_x']].round(3)

**How to read it.**

* `loc_x` large, `perfect_over_loc` near 1 → the whole gain is localization; the predictor is
  decoration on this primitive. That is the Array paper's permutation finding, sharpened: not just
  "the weights need not be accurate" but "the weights need not be there at all".
* `perfect_over_loc` clearly above `permuted_over_loc` → the predictor carries real structural
  information about which edges bear cross-triangles, and `structural_share` quantifies how much.
* `mindeg_over_loc` below 1 → the cheap Tonic-style predictor actively hurts here. Expected, and
  worth stating: min endpoint degree tracks *total* triangles, and on these projections most
  triangles are hyperedge-covered, so the heuristic aims the sampler at exactly the wrong edges.
  If this reproduces on real data it is a concrete negative transfer result for practitioners.
* `firstedge_x` vs `perfect_x` → replicates the whole-path-versus-first-edge experiment of §3 of the
  Array paper on the new primitive.

## 3. Synthetic scaling: does the speedup grow with $m$ here?

On the separable family (disjoint large hyperedges as copy-free cliques plus shallow cross gadgets)
the theory predicts a polynomially growing gain. This is the control that shows the machinery is
capable of an exponent when the structure permits — the contrast that makes the real-data result
meaningful rather than a null measurement.

In [ ]:
rowsS = []
for cs in [20, 30, 40, 60, 80]:
    P = CrossProjection(cliques_plus_cross(6, cs, 6 * cs, seed=1), max_size=200)
    r = P.report(); r['label'] = f'cliques(r={cs})'
    rowsS.append(r)
S = pd.DataFrame(rowsS)
b_loc = np.polyfit(np.log(S.m), np.log(S.loc_x), 1)[0]
b_all = np.polyfit(np.log(S.m), np.log(S.perfect_x), 1)[0]
print(f'localization grows as m^{b_loc:.3f};  localization+predictor as m^{b_all:.3f}')

plt.figure(figsize=(5.4,3.6))
plt.loglog(S.m, S.loc_x, 'o-', label=f'localization only (m^{b_loc:.2f})')
plt.loglog(S.m, S.perfect_x, 's-', label=f'+ perfect predictor (m^{b_all:.2f})')
if len(D):
    plt.scatter(D.m, D.perfect_x, c='crimson', marker='x', s=60, label='real hypergraphs')
plt.xlabel('m'); plt.ylabel('speedup over Fichtenberger-Peng'); plt.legend(fontsize=8)
plt.title('Separable family vs real data'); plt.tight_layout(); plt.show()
S[['label','m','n_cross','loc_x','perfect_x','perfect_over_loc','permuted_over_loc',
   'structural_share']].round(3)

## 4. Summary

In [ ]:
ALL = pd.concat([D.assign(kind='real'), S.assign(kind='synthetic')], ignore_index=True, sort=False)
ALL.to_csv('predcross_speedups.csv', index=False)
print('wrote predcross_speedups.csv')
print()
if len(D):
    print(f"real data: localization {D.loc_x.min():.1f}x - {D.loc_x.max():.1f}x, "
          f"predictor on top {D.perfect_over_loc.min():.2f}x - {D.perfect_over_loc.max():.2f}x, "
          f"structural share {D.structural_share.min():.2f} - {D.structural_share.max():.2f}")
ALL[['kind','label','m','n_cross','loc_x','perfect_x','perfect_over_loc',
     'permuted_over_loc','mindeg_over_loc','structural_share']].round(3)

## 5. What each outcome means for the DMKD submission

**If `perfect_over_loc` is large on real data** (say $>3\times$): the paper has a clean positive half
— a new higher-order primitive where predictions deliver a real constant-factor win — beside the
negative half (no exponent, ever) and the three lemmas. Strongest version.

**If `perfect_over_loc` hovers near 1** (the likelier outcome, given the overlapping structure we
already measured): the finding is that for higher-order triadic closure the predictor is redundant
and the sampler should simply be localized. That is still a positive, actionable result for a
practitioner — cheaper system, no model to train, no oracle to maintain — and it extends the
permutation control of the Array paper from "accuracy does not matter much" to "the weights do not
matter at all on this primitive". Pair it with `mindeg_over_loc < 1` and you have a concrete warning
against porting the standard triangle-counting heuristic to higher-order data.

Either way the submission needs the delta against the localization paper stated in its own
paragraph: that paper asks where the speedup comes from for ordinary triangles; this one asks
whether predictions can ever change the exponent, answers no with a structural mechanism, and
measures a new primitive on higher-order data.